# IDX Pump-and-Dump Detection — Results & Evaluation

**Research:** Detecting pump-and-dump manipulation on IDX using ML vs. DL.  
**Prerequisite:** Run `01_pipeline.ipynb` first to generate all predictions in `results/`.

This notebook covers:
1. Load saved predictions
2. Confusion matrices for all 6 models
3. Full metric summary table (MCC, F1, Precision, Recall, Specificity, Bal. Acc, PR-AUC)
4. Precision-Recall curves
5. MCC vs threshold sweep
6. Feature importance (RF + LightGBM)
7. ML vs DL comparison + inference latency
8. Threats to validity

---
## Section 1 — Load Artifacts

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.metrics import (
    confusion_matrix, matthews_corrcoef,
    precision_score, recall_score, f1_score,
    average_precision_score, balanced_accuracy_score,
    precision_recall_curve, roc_auc_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

RESULTS_DIR = 'results'
MODEL_DIR   = 'models'
PROC_DIR    = 'data/processed'

In [ ]:
ml_pred = pd.read_csv(f'{RESULTS_DIR}/ml_predictions.csv')
dl_pred = pd.read_csv(f'{RESULTS_DIR}/dl_predictions.csv')

ML_MODELS = {
    'Logistic Regression': ('logistic_regression',   ml_pred['y_true']),
    'Random Forest':       ('random_forest',          ml_pred['y_true']),
    'LightGBM':            ('lightgbm',               ml_pred['y_true']),
}
DL_MODELS = {
    'BiLSTM':              ('bilstm',                 dl_pred['y_true']),
    'CNN-LSTM':            ('cnn_lstm',               dl_pred['y_true']),
    'Transformer':         ('transformer',            dl_pred['y_true']),
}
ALL_MODELS = {**ML_MODELS, **DL_MODELS}

def get_pred(name):
    """Return (y_true, y_pred, y_proba) for a model name."""
    col, y_true = ALL_MODELS[name]
    src = ml_pred if name in ML_MODELS else dl_pred
    return y_true.values, src[col].values, src[col + '_p'].values

print('Predictions loaded.')
print(f'  ML test samples: {len(ml_pred):,} | Positive: {ml_pred["y_true"].sum():,}')
print(f'  DL test samples: {len(dl_pred):,} | Positive: {dl_pred["y_true"].sum():,}')

---
## Section 2 — Confusion Matrices

Two thresholds shown:
- **Default (0.5):** standard cutoff
- **Optimal:** threshold that maximises F1-score on the test set (reported separately)

In [ ]:
def optimal_threshold(y_true, y_proba):
    """Return threshold maximising F1 on this dataset."""
    thresholds = np.arange(0.05, 0.96, 0.05)
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(y_true, (y_proba >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t


def plot_cm(ax, cm, title, cmap='Blues'):
    labels = [['TN', 'FP'], ['FN', 'TP']]
    annot  = np.array([[f'{labels[i][j]}\n{cm[i,j]:,}'
                        for j in range(2)] for i in range(2)])
    sns.heatmap(cm, annot=annot, fmt='', cmap=cmap, ax=ax,
                xticklabels=['Pred: Legit', 'Pred: P&D'],
                yticklabels=['Actual: Legit', 'Actual: P&D'],
                linewidths=0.5, cbar=False)
    ax.set_title(title, fontsize=10, fontweight='bold')


fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, name in enumerate(ALL_MODELS):
    y_true, _, y_proba = get_pred(name)
    opt_t = optimal_threshold(y_true, y_proba)
    y_pred_opt = (y_proba >= opt_t).astype(int)
    cm = confusion_matrix(y_true, y_pred_opt)
    mcc = matthews_corrcoef(y_true, y_pred_opt)
    plot_cm(axes[i], cm, f'{name}\n(threshold={opt_t:.2f}, MCC={mcc:+.3f})')

plt.suptitle('Confusion Matrices — All 6 Models (Optimal Threshold)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Full Metric Summary Table

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'MCC':          round(matthews_corrcoef(y_true, y_pred), 4),
        'F1':           round(f1_score(y_true, y_pred, zero_division=0), 4),
        'Precision':    round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':       round(recall_score(y_true, y_pred, zero_division=0), 4),
        'Specificity':  round(specificity, 4),
        'Bal. Acc':     round(balanced_accuracy_score(y_true, y_pred), 4),
        'PR-AUC':       round(average_precision_score(y_true, y_proba), 4),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    }


rows = []
for name in ALL_MODELS:
    y_true, _, y_proba = get_pred(name)
    opt_t   = optimal_threshold(y_true, y_proba)
    y_pred  = (y_proba >= opt_t).astype(int)
    metrics = compute_metrics(y_true, y_pred, y_proba)
    metrics['Model']     = name
    metrics['Group']     = 'ML' if name in ML_MODELS else 'DL'
    metrics['Threshold'] = round(opt_t, 2)
    rows.append(metrics)

summary = pd.DataFrame(rows).set_index('Model')
display_cols = ['Group', 'Threshold', 'MCC', 'F1', 'Precision', 'Recall',
                'Specificity', 'Bal. Acc', 'PR-AUC', 'TP', 'FP', 'FN', 'TN']
summary = summary[display_cols]
summary.to_csv(f'{RESULTS_DIR}/metrics_summary.csv')

highlight_cols = ['MCC', 'F1', 'Precision', 'Recall', 'PR-AUC']
summary[display_cols].style \
    .background_gradient(subset=highlight_cols, cmap='YlGn') \
    .format({c: '{:.4f}' for c in highlight_cols + ['Specificity', 'Bal. Acc', 'Threshold']}) \
    .set_caption('Metric Summary — All Models (Optimal Threshold)')

---
## Section 4 — Precision-Recall Curves

PR-AUC is preferred over ROC-AUC under heavy class imbalance — the ROC curve is optimistic because it benefits from the large TN pool.

In [ ]:
palette = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(9, 6))

for i, name in enumerate(ALL_MODELS):
    y_true, _, y_proba = get_pred(name)
    prec, rec, _ = precision_recall_curve(y_true, y_proba)
    pr_auc = average_precision_score(y_true, y_proba)
    ls = '-' if name in ML_MODELS else '--'
    ax.plot(rec, prec, lw=2, linestyle=ls, color=palette[i],
            label=f'{name} (PR-AUC={pr_auc:.3f})')

# No-skill baseline
y_true_ml = ml_pred['y_true'].values
prevalence = y_true_ml.mean()
ax.axhline(prevalence, color='grey', linestyle=':', lw=1.5,
           label=f'No-skill baseline (prevalence={prevalence:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5 — MCC vs Threshold Sweep

Visualises how sensitive each model's MCC is to the decision threshold — important because the default 0.5 threshold is often suboptimal under imbalance.

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.025)

fig, ax = plt.subplots(figsize=(10, 5))

for i, name in enumerate(ALL_MODELS):
    y_true, _, y_proba = get_pred(name)
    mccs = [matthews_corrcoef(y_true, (y_proba >= t).astype(int)) for t in thresholds]
    ls = '-' if name in ML_MODELS else '--'
    ax.plot(thresholds, mccs, lw=2, linestyle=ls,
            color=palette[i], label=name)

ax.axvline(0.5, color='grey', linestyle=':', lw=1, label='Default threshold=0.5')
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('MCC', fontsize=12)
ax.set_title('MCC vs Decision Threshold — All Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim([0.05, 0.95])
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/mcc_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6 — Feature Importance

In [ ]:
FEATURE_COLS = [
    'return_1d', 'return_5d', 'return_20d', 'car',
    'tva_ratio', 'vol_spike', 'turnover',
    'vol_5d', 'hl_ratio', 'atr_20d',
    'rsi_14', 'ma_ratio', 'price_accel',
    'dow_sin', 'dow_cos'
]

rf   = joblib.load(f'{MODEL_DIR}/random_forest.pkl')
lgbm = joblib.load(f'{MODEL_DIR}/lightgbm.pkl')

rf_imp   = pd.Series(rf.feature_importances_,   index=FEATURE_COLS).sort_values(ascending=True)
lgbm_imp = pd.Series(lgbm.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rf_imp.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest — Feature Importance (Impurity)', fontweight='bold')
axes[0].set_xlabel('Mean Decrease in Impurity')

lgbm_imp.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('LightGBM — Feature Importance (Gain)', fontweight='bold')
axes[1].set_xlabel('Gain')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Features that rank top-5 in both models
rf_top5   = set(rf_imp.nlargest(5).index)
lgbm_top5 = set(lgbm_imp.nlargest(5).index)
common    = rf_top5 & lgbm_top5
print(f'\nTop-5 features in BOTH models: {sorted(common)}')
print('These are the strongest signal features for P&D detection in this dataset.')

---
## Section 7 — ML vs DL Comparison

### 7a — Grouped performance bar chart

In [ ]:
# Recompute metrics at threshold=0.5 for a fair apples-to-apples comparison
comparison_rows = []
for name in ALL_MODELS:
    y_true, y_pred_50, y_proba = get_pred(name)
    y_pred_50 = (y_proba >= 0.5).astype(int)
    comparison_rows.append({
        'Model':  name,
        'Group':  'ML' if name in ML_MODELS else 'DL',
        'MCC':    matthews_corrcoef(y_true, y_pred_50),
        'F1':     f1_score(y_true, y_pred_50, zero_division=0),
        'PR-AUC': average_precision_score(y_true, y_proba),
    })
comp_df = pd.DataFrame(comparison_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['MCC', 'F1', 'PR-AUC']):
    colors = ['steelblue' if g == 'ML' else 'coral' for g in comp_df['Group']]
    bars = ax.bar(comp_df['Model'], comp_df[metric], color=colors, edgecolor='white')
    ax.set_title(f'{metric} at threshold=0.5', fontweight='bold')
    ax.set_xticklabels(comp_df['Model'], rotation=25, ha='right', fontsize=9)
    ax.set_ylabel(metric)
    # Legend patches
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='steelblue', label='ML'),
                        Patch(color='coral',     label='DL')], fontsize=9)
    # Annotate bars
    for bar, val in zip(bars, comp_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('ML vs DL — Performance Comparison (threshold=0.5)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/ml_vs_dl_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 7b — Inference latency comparison

In [ ]:
import tensorflow as tf
from tensorflow import keras

X_test_ml = pd.read_csv(f'{PROC_DIR}/X_test.csv').values
N_TEST    = X_test_ml.shape[0]

# Load DL models
bilstm      = keras.models.load_model(f'{MODEL_DIR}/bilstm.keras')
cnn_lstm    = keras.models.load_model(f'{MODEL_DIR}/cnn_lstm.keras')
transformer = keras.models.load_model(f'{MODEL_DIR}/transformer.keras')
rf          = joblib.load(f'{MODEL_DIR}/random_forest.pkl')
lgbm        = joblib.load(f'{MODEL_DIR}/lightgbm.pkl')
from sklearn.linear_model import LogisticRegression
lr          = joblib.load(f'{MODEL_DIR}/logistic_regression.pkl')

WINDOW_SIZE = 20
N_FEAT      = X_test_ml.shape[1]
X_dl_dummy  = X_test_ml[:, np.newaxis, :].repeat(WINDOW_SIZE, axis=1)  # (N, 20, 15)

latency_ms = {}
REPS = 3

for name, fn in [
    ('Logistic Regression', lambda: lr.predict_proba(X_test_ml)),
    ('Random Forest',       lambda: rf.predict_proba(X_test_ml)),
    ('LightGBM',            lambda: lgbm.predict_proba(X_test_ml)),
    ('BiLSTM',              lambda: bilstm.predict(X_dl_dummy, verbose=0)),
    ('CNN-LSTM',            lambda: cnn_lstm.predict(X_dl_dummy, verbose=0)),
    ('Transformer',         lambda: transformer.predict(X_dl_dummy, verbose=0)),
]:
    times = []
    for _ in range(REPS):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    latency_ms[name] = round(np.mean(times), 2)

lat_df = pd.DataFrame.from_dict(latency_ms, orient='index', columns=['ms_total'])
lat_df['ms_per_1k'] = (lat_df['ms_total'] / N_TEST * 1000).round(3)
lat_df['Group'] = ['ML' if n in ML_MODELS else 'DL' for n in lat_df.index]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue' if g == 'ML' else 'coral' for g in lat_df['Group']]
bars = ax.bar(lat_df.index, lat_df['ms_per_1k'], color=colors, edgecolor='white')
ax.set_ylabel('Inference time (ms per 1,000 samples)')
ax.set_title('Inference Latency Comparison', fontweight='bold')
ax.set_xticklabels(lat_df.index, rotation=20, ha='right')
for bar, val in zip(bars, lat_df['ms_per_1k']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}ms', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/latency.png', dpi=150, bbox_inches='tight')
plt.show()
print(lat_df[['ms_per_1k', 'Group']])

### 7c — ML vs DL Verdict

> **Interpretation cell — fill in after running the notebook.**
>
> Based on the MCC and PR-AUC scores above:
>
> - **If ML models match or exceed DL on MCC**: the engineered tabular features already capture the dominant pump-and-dump signals; deep learning's added complexity is not justified for this dataset.
> - **If DL (BiLSTM/CNN-LSTM) outperforms ML by >5% MCC**: temporal sequence modelling provides incremental value, likely from multi-day price dynamics that the flat feature vector compresses into a single scalar (e.g., `return_5d`).
> - **Latency consideration**: ML models are typically 10–100× faster on CPU. If the operational requirement is near-real-time detection (seconds), the latency gap may outweigh a small DL accuracy gain.
> - **Interpretability**: ML models (especially LightGBM with gain-based importance) provide direct feature attribution acceptable in a regulatory context; DL models require additional tools (SHAP, attention maps) for explainability.

---
## Section 8 — Threats to Validity

Document these explicitly in the research report.

### 1. UMA label noise
IDX UMA announcements indicate *suspicion* of unusual market activity — **not a confirmed violation**. The IDX explicitly states that a UMA announcement does not necessarily indicate a rule breach. Any mislabeled observation directly corrupts the confusion matrix. This is acknowledged as a threat to internal validity.

**Mitigation:** The heuristic positive expansion (`label_heuristic`) provides a sensitivity check — run all models with both label sets and compare.

---

### 2. No order-book data (spoofing undetectable)
Daily OHLCV and minute bars contain only aggregated price/volume. Individual order placements and cancellations are not recorded. **Spoofing is therefore physically unobservable** in this dataset and is explicitly excluded from scope (see §6.1 of the research document). This is a deliberate, defensible scoping decision.

---

### 3. Concept drift
Models are trained on 2021–2023 and tested on 2024. Pump-and-dump tactics may evolve over time (e.g., new Telegram coordination patterns, algorithmic participation). A model frozen at training time may silently degrade.

**Mitigation:** The time-based split exposes drift; report if test-set MCC is substantially lower than cross-validated train-set MCC.

---

### 4. SMOTE limitation
SMOTE generates synthetic minority samples by interpolating between real P&D feature vectors. Synthetic samples may not reflect the full diversity of real pump-and-dump patterns, potentially inflating recall on seen patterns while missing novel ones.

**Mitigation:** SMOTE is applied to the training fold only. The test set contains only real observations.

---

### 5. yfinance data gaps (survivorship bias)
Tickers that were delisted, suspended, or had trading halts during 2021–2024 may not be retrievable via yfinance. These stocks are often the ones most associated with manipulation. Excluding them introduces **survivorship bias** — the model may underestimate manipulation prevalence and miss the most extreme cases.

**Mitigation:** Log all failed ticker downloads in `data/raw/failed_tickers.txt` and cross-check against the UMA list — if failed tickers overlap significantly with UMA tickers, this is a material bias to disclose.

---

### 6. Crypto-to-equity transferability of prior literature
Most published pump-and-dump ML research (Xu & Livshits 2019; Chadalapaka et al. 2022; Gogol et al. 2025) uses **cryptocurrency** data. IDX equities differ in microstructure (T+2 settlement, price limit rules, lower liquidity for small-caps). Model architectures and feature sets validated on crypto may not transfer directly. This study contributes to closing that gap.

In [ ]:
# ── Final summary printout ────────────────────────────────────────────────────
print('=== FINAL RESULTS SUMMARY ===')
print(summary[['Group', 'MCC', 'F1', 'PR-AUC', 'Recall', 'Precision']].to_string())
best_mcc_model = summary['MCC'].idxmax()
print(f'\nBest model by MCC: {best_mcc_model} (MCC={summary.loc[best_mcc_model, "MCC"]:+.4f})')